In [ ]:

from config.config import *
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, BooleanType

# Load dimension tables
dim_country = spark.read.format("delta").load(f"{gold_path}/dim_country")
dim_date = spark.read.format("delta").load(f"{gold_path}/dim_date")

# Helper to attach date/country keys
def with_keys(df, year_col="year", month_col="month", geo_col="geo_code"):
    return df.join(dim_country, on=[dim_country["country_iso3"] == df[geo_col]], how="left")              .join(dim_date, on=[dim_date["year"] == df[year_col], dim_date["month"] == df[month_col]], how="left")

# ---------- fact_tourism_arrivals ----------
arr_schema = StructType([
    StructField("arrival_key", IntegerType(), True), StructField("country_key", IntegerType(), True),
    StructField("geo_code", StringType(), True), StructField("country_name", StringType(), True),
    StructField("year", IntegerType(), True), StructField("month", IntegerType(), True),
    StructField("date_key", IntegerType(), True), StructField("frequency", StringType(), True),
    StructField("source_system", StringType(), True), StructField("indicator_code", StringType(), True),
    StructField("indicator_label", StringType(), True), StructField("visitor_category", StringType(), True),
    StructField("residence_category", StringType(), True), StructField("accommodation_category", StringType(), True),
    StructField("original_value", DoubleType(), True), StructField("original_unit", StringType(), True),
    StructField("standardized_value", DoubleType(), True), StructField("standardized_unit", StringType(), True),
    StructField("observation_status", StringType(), True), StructField("aggregation_level", StringType(), True),
    StructField("is_country", BooleanType(), True), StructField("is_primary_period", BooleanType(), True),
])

rows = []
# UN inbound arrivals (annual). Reconstruct original unit as "thousand trips" and standardize to persons.
un_arr = spark.read.format("delta").load(f"{silver_path}/un_tourism_inbound_arrivals")
visitor_map = {
    "INBD_TRIP_TOTL_TOTL_TOUR": "Overnight visitors (tourists)",
    "INBD_TRIP_TOTL_TOTL_VSTR": "Visitors (total)",
    "INBD_TRIP_TOTL_TOTL_EXCR": "Same-day visitors (excursionists)",
    "INBD_TRIP_TOTL_CRUS_EXCR": "Cruise-ship same-day visitors",
}
for r in un_arr.collect():
    rows.append((None, r["reporter_country_iso3"] and dim_country.where(dim_country["iso3"] == r["reporter_country_iso3"]).first()["country_key"],
                 r["reporter_area"], None, r["year"], None, r["year"], "A", "UN_TOURISM",
                 r["indicator_code"], r["indicator_label"], visitor_map.get(r["indicator_code"], "Visitors (total)"),
                 None, None, r["arrivals"] / 1000.0 if r["arrivals"] is not None else None, "thousand trips",
                 r["arrivals"], "persons", r["observation_flag"], "Country", True, 2012 <= r["year"] <= 2019))

# World Bank (annual)
wb = spark.read.format("delta").load(f"{silver_path}/world_bank_international_arrivals")
for r in wb.filter(F.col("country_iso3").isNotNull()).collect():
    key = dim_country.where(dim_country["iso3"] == r["country_iso3"]).first()["country_key"]
    rows.append((None, key, r["country_code"], r["country_name"], r["year"], None, r["year"], "A", "WORLD_BANK",
                 r["indicator_code"], r["indicator_name"], "International arrivals", None, None,
                 r["arrivals"], "number of arrivals", r["arrivals"], "persons", None, "Country", True, 2012 <= r["year"] <= 2019))

# Eurostat arrivals (monthly). Only unit_code = NR are counts.
estat_arm = spark.read.format("delta").load(f"{silver_path}/estat_tour_occ_arm")
for r in estat_arm.filter(F.col("unit_code") == "NR").collect():
    key = dim_country.where(dim_country["iso3"] == r["country_iso3"]).first()["country_key"] if r["country_iso3"] else None
    rows.append((None, key, r["geo_code"], None, r["period"].year, r["period"].month, r["period"].year * 100 + r["period"].month,
                 "M", "EUROSTAT", r["activity_code"], r["geo_code"], "International arrivals", None, None,
                 r["tourism_value"], "number of arrivals", r["tourism_value"], "persons", r["observation_flag"],
                 "Country", key is not None, 2012 <= r["period"].year <= 2019))

fact_tourism_arrivals = spark.createDataFrame(rows, arr_schema)
fact_tourism_arrivals.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/fact_tourism_arrivals")
print("fact_tourism_arrivals rows:", fact_tourism_arrivals.count())


In [ ]:

from config.config import *
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, BooleanType

# fact_nights_spent: Eurostat monthly nights spent (unit_code = NR)
nights_schema = StructType([
    StructField("nights_key", IntegerType(), True), StructField("country_key", IntegerType(), True),
    StructField("geo_code", StringType(), True), StructField("year", IntegerType(), True),
    StructField("month", IntegerType(), True), StructField("date_key", IntegerType(), True),
    StructField("residence_category", StringType(), True), StructField("activity_category", StringType(), True),
    StructField("source_system", StringType(), True), StructField("original_value", DoubleType(), True),
    StructField("original_unit", StringType(), True), StructField("standardized_value", DoubleType(), True),
    StructField("standardized_unit", StringType(), True), StructField("observation_status", StringType(), True),
    StructField("is_primary_period", BooleanType(), True),
])
dim_country = spark.read.format("delta").load(f"{gold_path}/dim_country")
rows = []
estat_nim = spark.read.format("delta").load(f"{silver_path}/estat_tour_occ_nim")
for r in estat_nim.filter(F.col("unit_code") == "NR").collect():
    key = dim_country.where(dim_country["iso3"] == r["country_iso3"]).first()["country_key"] if r["country_iso3"] else None
    rows.append((None, key, r["geo_code"], r["period"].year, r["period"].month, r["period"].year * 100 + r["period"].month,
                 r["residence_category"], r["activity_code"], "EUROSTAT", r["nights_spent"], "nights",
                 r["nights_spent"], "nights", r["observation_flag"], 2012 <= r["period"].year <= 2019))
fact_nights_spent = spark.createDataFrame(rows, nights_schema)
fact_nights_spent.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/fact_nights_spent")
print("fact_nights_spent rows:", fact_nights_spent.count())


In [ ]:

from config.config import *
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, BooleanType

# fact_hotel_performance: UN accommodation split by indicator/unit
hotel_schema = StructType([
    StructField("hotel_perf_key", IntegerType(), True), StructField("country_key", IntegerType(), True),
    StructField("geo_code", StringType(), True), StructField("year", IntegerType(), True),
    StructField("month", IntegerType(), True), StructField("date_key", IntegerType(), True),
    StructField("source_system", StringType(), True), StructField("indicator_code", StringType(), True),
    StructField("indicator_label", StringType(), True), StructField("accommodation_metric", StringType(), True),
    StructField("original_value", DoubleType(), True), StructField("original_unit", StringType(), True),
    StructField("standardized_value", DoubleType(), True), StructField("standardized_unit", StringType(), True),
    StructField("observation_status", StringType(), True), StructField("is_primary_period", BooleanType(), True),
])
dim_country = spark.read.format("delta").load(f"{gold_path}/dim_country")
rows = []
un_ac = spark.read.format("delta").load(f"{silver_path}/un_tourism_accommodation_hotels")
metric_map = {
    "CPTY_ACCM_HOTL_ESTB": "Establishments",
    "CPTY_ACCM_HOTL_BEDS": "Bed places",
    "CPTY_ACCM_HOTL_ROOM": "Rooms",
    "CPTY_ACCM_HOTL_BEDS_RATE": "Bed-place rate",
    "CPTY_ACCM_HOTL_ROOM_RATE": "Room rate",
    "CPTY_ACCM_HOTL_STAY_AVRG": "Average length of stay",
}
for r in un_ac.collect():
    key = dim_country.where(dim_country["reporter_area"] == r["reporter_area"]).first()["country_key"] if r["reporter_area"] else None
    rows.append((None, key, r["reporter_area"], r["year"], None, r["year"], "UN_TOURISM",
                 r["indicator_code"], r["indicator_label"], metric_map.get(r["indicator_code"], r["indicator_code"]),
                 r["hotel_capacity"], r["unit"], r["hotel_capacity"], r["unit"], r["observation_flag"], 2012 <= r["year"] <= 2019))
fact_hotel_performance = spark.createDataFrame(rows, hotel_schema)
fact_hotel_performance.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/fact_hotel_performance")
print("fact_hotel_performance rows:", fact_hotel_performance.count())


In [ ]:

from config.config import *
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, BooleanType

# fact_hotel_occupancy: Eurostat monthly occupancy (BEDPL/BEDRM separate)
occ_schema = StructType([
    StructField("occupancy_key", IntegerType(), True), StructField("country_key", IntegerType(), True),
    StructField("geo_code", StringType(), True), StructField("year", IntegerType(), True),
    StructField("month", IntegerType(), True), StructField("date_key", IntegerType(), True),
    StructField("accommodation_unit", StringType(), True), StructField("occupancy_rate", DoubleType(), True),
    StructField("unit", StringType(), True), StructField("source_system", StringType(), True),
    StructField("observation_status", StringType(), True), StructField("is_primary_period", BooleanType(), True),
])
dim_country = spark.read.format("delta").load(f"{gold_path}/dim_country")
rows = []
estat_mnor = spark.read.format("delta").load(f"{silver_path}/estat_tour_occ_mnor")
for r in estat_mnor.filter(F.col("unit_code") == "PC").collect():
    key = dim_country.where(dim_country["iso3"] == r["country_iso3"]).first()["country_key"] if r["country_iso3"] else None
    rows.append((None, key, r["geo_code"], r["period"].year, r["period"].month, r["period"].year * 100 + r["period"].month,
                 r["accommodation_unit"], r["occupancy_rate"], "PC", "EUROSTAT", r["observation_flag"], 2012 <= r["period"].year <= 2019))
fact_hotel_occupancy = spark.createDataFrame(rows, occ_schema)
fact_hotel_occupancy.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/fact_hotel_occupancy")
print("fact_hotel_occupancy rows:", fact_hotel_occupancy.count())
